In [ ]:
# importing stuff here 
using Distributions
using DataFrames
using Statistics
using Optim

# Simulation parameters
n_market = 50
n_product = 6
n_consumer = 1000
true_alpha = 1.0
true_beta = 2.0
true_sigma = 1.0
core_char = [1.2, 0.5, -0.8, 0.3, 1.0, -0.6]


# new parameters
n_all_firm = 10 #maximum number of firms
n_chars = 2 #number of prod characteristics
n_consumer = 200 #number of consumer/market 

n_prod_char = 2;                  # Product characteristics
n_market = 100;                # Number of markets
β = [.5, 2, -1];        # Preferences
varζ = 5;               # Variance of the random taste
rangeJ = [2, 6];        # Min and max firms per market
varX = 1;               # Variance of X
varξ = 2;               # Variance of xi



In [2]:
rand(Normal(0, 5), 4+1, 2)   

5×2 Matrix{Float64}:
  9.53078  -7.56962
  3.78281   1.56165
 -1.43874   8.3339
 -5.34186  -0.474453
 -1.10045  -3.32416

In [ ]:
# simulation functions 
function demand(p, char, β, ξ, beta_var)
    """Compute demand in terms of market share"""
    # print("beta_var", beta_var)
    δ = [char p] * (β .+ beta_var)
    # print("here first")
    # print("this is ok")
    # print("calculated delta", δ)
    δ_0 = zeros(1, size(beta_var,2))
    u = [δ; δ_0] + ξ #sum of utilities
    # print("here!")
    exp_u = exp.(u) #exponent every terms
    q = mean(exp_u ./ sum(exp_u, dims=1), dims=2) #suming over utilities for market share 
    return q[1:end-1], q[end]
end;

function profits(p, c, X, β, ξ, beta_var) #return company-profit for a given setting 
    """Compute profits"""
    q, _ = demand(p, X, β, ξ, beta_var)            # Compute demand
    profit = (p - c) .* q                       # Compute profits
    return profit 
end;

function profits_j(pj, j, p, c, X, β, ξ, ζ)
    """Compute profits of firm j"""
    p[j] = pj                               # Insert price of firm j
    pr = profits(p, c, X, β, ξ, ζ)          # Compute profits
    return pr[j]
end;

function draw_data(n_firm_max, n_chars, n_consumer, nonlinear)::Tuple
    """Draw data for one market"""
    J_ = rand(2:n_firm_max)              # Number of firms (products)
    # print("no of firms is ", n_firm_max)
    # println("drew", J_)
    X_ = rand(Normal(0,1), J_, n_chars)         # Product characteristics
    ξ_ = rand(Normal(0,1), J_+1, n_consumer)         # Product-level utility shocks ~ Normal(0,1)
    # Consumer-product-level preference shocks
    ζ_ = [rand(Normal(0,1), 1, n_consumer) * nonlinear; zeros(n_chars,n_consumer)]
    # ζ_ have only heterogeneity in price coefficient, but NOT in product chars coefficients
    w_ = rand(Uniform(0,1), J_)                # Cost shifters per firm/market
    ω_ = rand(Uniform(0,1), J_)                # Cost shocks per firm/market
    c_ = w_ + ω_                                # Cost
    # print("here now")
    j_ = sort(sample(1:n_all_firm, J_, replace=false))   # Subset of firms
    # print("zeta", ζ_, dim(ζ_))
    # display(ζ_)
    return X_, ξ_, ζ_, w_, c_, j_
end;

# draw_data(10,2,3,0.5)

function equilibrium(cost, X, β, ξ, ζ)::Vector
    """Compute equilibrium prices and profits"""
    p = 2 .* c; #starting guess of price vector 
    dist = 1;
    iter = 0;

    # Iterate until convergence
    while (dist > 1e-8) && (iter<1000)
        # Compute best reply for each firm
        p_old = copy(p);
        for j=1:length(p)
            obj_fun(pj) = - profits_j(pj[1], j, p, c, X, β, ξ, ζ);
            optimize(x -> obj_fun(x), [1.0], LBFGS());
        end
        # Update distance
        dist = max(abs.(p - p_old)...);
        iter += 1;
    end
    return p
end;


function compute(n_consumer, n_firm_max, β, nonlinear)
    """Compute equilibrium one market"""
    # Initialize variables
    K = size(β, 1) - 1 #number of characteristics 
    X_, ξ_, ζ_, w_, c_, j_ = draw_data(n_firm_max, K, n_consumer, nonlinear)

    # Compute equilibrium
    p_ = equilibrium(c_, X_, β, ξ_, ζ_)    # Equilibrium prices
    q_, q0 = demand(p_, X_, β, ξ_, ζ_)     # Demand with shocks
    pr_ = (p_ - c_) .* q_                       # Profits

    # Save to data
    q0_ = ones(length(j_)) .* q0
    df = DataFrame(j=j_, w=w_, p=p_, q=q_, q0=q0_, pr=pr_)
    for k=1:K
      df[!,"x$k"] = X_[:,k]
      df[!,"z$k"] = sum(X_[:,k]) .- X_[:,k]
    end
    return df
end;

function simulate_data(n_consumer, n_firm_max, β, nonlinear)
    """Simulate full dataset"""
    df = compute(n_consumer, n_firm_max, β, nonlinear)
    df[!, "t"] = ones(nrow(df)) * 1
    for t=2:n_market
        df_temp = compute(n_consumer, n_firm_max, β, nonlinear)
        df_temp[!, "t"] = ones(nrow(df_temp)) * t
        append!(df, df_temp)
    end
    CSV.write("../data/blp.csv", df)
    return df
end;

print(compute(10,10,[1 1 1], 0.5))


3×3 Matrix{Float64}:
 0.417265  0.677533  0.591931
 0.0       0.0       0.0
 0.0       0.0       0.0

MethodError: MethodError: Cannot `convert` an object of type Nothing to an object of type Tuple
The function `convert` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  (::Type{T})(::Any) where T<:Tuple
   @ Base tuple.jl:455
  convert(::Type{T}, !Matched::T) where T<:Tuple
   @ Base essentials.jl:600
  convert(::Type{T}, !Matched::T) where T
   @ Base Base.jl:126
  ...


In [4]:
# creating the simulated dataset 
all_markets = DataFrame[]

for market in 1:n_market
    prod_char = deepcopy(core_char)
    prod_char = (prod_char .- mean(prod_char)) ./ std(prod_char)

    cost_shifter = rand(Normal(0, 2), n_product)
    price = price_gen(prod_char, cost_shifter)
    price = (price .- mean(price)) ./ std(price)

    utility_matrix = utility_gen([true_alpha, true_beta], true_sigma, prod_char, price, n_consumer)
    choice_probs = choice_prob(utility_matrix)

    market_share = vec(sum(choice_probs, dims=2)) ./ n_consumer

    instrument_cost = rand(Normal(0, 2), n_product)
    instrument_cost = (instrument_cost .- mean(instrument_cost)) ./ std(instrument_cost)

    instrument_price = sum(price) .- price
    instrument_price = (instrument_price .- mean(instrument_price)) ./ std(instrument_price)

    instrument_char = sum(prod_char) .- prod_char
    instrument_char = (instrument_char .- mean(instrument_char)) ./ std(instrument_char)

    df = DataFrame(
        market_id = fill(market, n_product),
        product_id = 0:n_product-1,
        prod_char = prod_char,
        price = price,
        share = market_share,
        true_delta = true_beta .* prod_char .- true_alpha .* price,
        share_not_buy = fill(1 - sum(market_share), n_product),
        instrument_cost = instrument_cost,
        instrument_char = instrument_char,
        instrument_price = instrument_price
    )

    push!(all_markets, df)
end

export_dataset = vcat(all_markets...)
CSV.write("bem.csv", export_dataset)

UndefVarError: UndefVarError: `price_gen` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
# η: how increases in coal consumption correspond to increases in permit prices 

